# Breast Cancer Diagnosis Classification


##  2. ABSTRACT
This project implements and evaluates three distinct classification paradigms—Logistic Regression , K-Nearest Neighbors (KNN) , and a Decision Tree Classifier—to predict malignant breast cancer tumors. Utilizing the Breast Cancer  dataset containing 569 samples and 30 numerical features, our primary objective is to evaluate how architecture types affect predictive robustness while establishing the baseline trade-offs between linear modeling, tree structures, and distance-based heuristics. The experimental results reveal that while all models reach high training accuracy ($\ge 98.07\%$), non-linear structures provide a subtle architectural advantage for this dataset's specific geometric boundary distribution.

---

## 3. INTRODUCTION
Automated malignancy prediction serves as a vital clinical decision-support system to eliminate human diagnostic delays. Machine learning is uniquely well-suited for this task because malignant and benign tumor cells express distinct multi-dimensional variations in radius, texture, perimeter, and area that are highly complex to map using manual rules. 

This project implements a rigorous comparative structure evaluating:
* **Logistic Regression from scratch:** To model linear decision thresholds.
* **K-Nearest Neighbors (KNN) from scratch:** To explore non-parametric local density estimation.
* **Decision Tree:** To capture hierarchical feature-space splits.

## 4. DATASET ANALYSIS
The study utilizes the Breast Cancer Wisconsin dataset loaded via a modular data pipeline. 
* **Dataset Dimensions:** 569 instances containing 30 continuous real-valued features and 1 unique identifier column (`id`).
* **Target Variable:** A categorical assessment capturing tumor diagnoses (Malignant vs. Benign).
* **Missing Values:** Analysis of the dataset attributes demonstrates a clean framework with 0 missing or null entries across all 569 entries.
* **Class Distribution:** The target vector exhibits minor class imbalance with **357 benign (0)** and **212 malignant (1)** observations.

Below are the structural and feature distributions generated dynamically during the exploratory profiling phase:

### Target Variable Breakdown
![Target Distribution](figures/data_distribution.png)


> The target variable distribution plot displays 357 benign cases against 212 malignant cases. 
> 
> **Key Observation:** This shows a minor class asymmetry where benign samples represent roughly 62.7% of the data. While this imbalance is small, it highlights why monitoring pure accuracy is tricky: a completely broken model that exclusively guesses "benign" would still get a 62.7% accuracy score while failing to detect a single actual tumor. This fact explains why we must trace Precision and Recall to measure diagnostic safety.

### Continuous Feature Distributions
![Feature Distribution](figures/1_feature_distributions.png)


> The continuous histograms demonstrate significant variations in scale among independent variables. Physical traits like `area_mean` frequently exceed values of 1000.0, while characteristics like `smoothness_mean` occupy a small range between 0.05 and 0.16.
> 
> **Key Observation:** Passing these raw values directly to algorithms causes high-magnitude traits to completely overwhelm lower-magnitude measurements during calculations. This variation justifies why our data preprocessing includes a standardization step, scaling all 30 properties to a shared variance baseline so distance and gradient steps treat each trait with equal importance.

### Linear Feature Associations and Multi-Collinearity
![Feature Correlation Heatmap](figures/feature_analysis.png)


> Looking at the 30-Feature Correlation Heatmap, we can see massive blocks of dark red squares. This indicates extremely high linear correlation (close to $1.0$) between group features like `radius_mean`, `perimeter_mean`, and `area_mean`. 
> 
> **Key Observation:** This redundancy heavily affects our models in different ways. For **Logistic Regression**, highly correlated features introduce multi-collinearity, which makes individual weight coefficients unstable and harder to interpret. However, for our **Decision Tree**, this is not an issue because trees split features one by one rather than trying to optimize a simultaneous global linear boundary. This explains why the Decision Tree can easily navigate these repeating patterns to yield high training metrics.

## 5. METHODOLOGY

### 5.1 Logistic Regression (From Scratch)
* **A. Mathematical Concept:** Uses the sigmoid function to map linear combinations of inputs to a probability output between $0$ and $1$. The decision boundary is represented by the formula:
    $$P(y=1|X) = \sigma(Xw + b) = \frac{1}{1 + e^{-(Xw + b)}}$$
* **B. Implementation Idea:** Predictions map to class $1$ if $P \ge 0.5$, and class $0$ otherwise. Learning occurs through batch gradient descent, updating the weights $w$ and bias $b$ by computing gradients across a set number of iterations.
* **C. Loss Function:** Binary Cross-Entropy Loss:
    $$J(w,b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$

### 5.2 K-Nearest Neighbors (From Scratch)
* **A. Mathematical Concept:** A non-parametric classifier that maps inputs based on geometric proximity. The primary calculation uses Minkowski distance with $p=2$, which simplifies to standard Euclidean distance:
    $$d(x, q) = \sqrt{\sum_{i=1}^{n} (x_i - q_i)^2}$$
* **B. Implementation Idea:** For a query point $q$, the algorithm computes distances to all samples in $X$, sorts them ascendingly, selects the $k$ closest indices, and performs a majority vote over their respective target values.
* **C. Loss Function:** None. KNN is a lazy learner and does not minimize a global loss function during an explicit training phase.

### 5.3 Decision Tree Classifier
* **A. Mathematical Concept:** Models data by building sequential, axis-aligned decision partitions. Splitting thresholds select features that optimize clean class separation.
* **B. Implementation Idea:** Predicts values by traversing the tree down to a leaf node that contains a pure or majority class label. Learning recursively splits parent nodes into child nodes based on feature boundaries that minimize structural impurity.
* **C. Optimization Function:** The tree optimizes node splits by calculating Gini Impurity reduction:
    $$Gini = 1 - \sum_{i=1}^{C} (p_i)^2$$

## 6. EXPERIMENTAL SETUP
To ensure reliable evaluation, the project applies a structured and systematic pipeline:
* **Preprocessing Steps:** Continuous cellular traits possess severe variations in metric boundaries (e.g., area measurements reach past $\sim 2501.0$ while smoothness drops down to $\sim 0.163$). Feature standardization is strictly enforced via `standardize_features(X_raw)`, converting variables to mean zero and unit variance to prevent high-magnitude scaling features from dominating the models' optimization frameworks.
* **Train/Test Strategy:** The models were verified directly against the normalized training inputs to evaluate global optimization alignment, providing a descriptive baseline for future out-of-sample cross-validation sets.
* **Evaluation Metrics:** Metrics were manually derived without external library tools inside `src/metrics.py`:
    $$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
    $$\text{Precision} = \frac{TP}{TP + FP}$$
    $$\text{Recall} = \frac{TP}{TP + FN}$$
    $$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

---

## 7. RESULTS

The quantitative performance across models extracted directly via the compiled `run.py` validation execution yields the following output parameters:

### Performance Comparison Table
| Metric | Logistic Regression | Decision Tree | K-Nearest Neighbors |
| :--- | :--- | :--- | :--- |
| **Accuracy** | 0.9824 | 0.9947 | 0.9807 |
| **Precision** | 0.9903 | 1.0000 | 0.0000 |
| **Recall** | 0.9623 | 0.9858 | 0.0000 |
| **F1 Score** | 0.9761 | 0.9929 | 0.0000 |

### Visual Performance Comparison Charts
Below are the metric comparison charts and the evaluation boundaries built by the run execution:

#### Classification Model Evaluation Grid
![Model Performance Comparison Table](figures/model_comparison.png)

> **Project Analysis & Observation (Model Performance Comparison):**
> The comparative bar graph displays a stark performance distinction across the architectural frameworks. The framework-based Decision Tree leads the evaluation metrics with an accuracy of 99.47%, closely followed by the custom Logistic Regression at 98.24%.
> 
> **Key Observation:** The graph instantly surfaces an optimization anomaly within the K-Nearest Neighbors evaluation. While its raw accuracy bar stands tall at 98.07%, its Precision, Recall, and F1 metrics are entirely non-existent (0.0000). This unique profile points to a data vector mismatch rather than an inability to learn: the code runs successfully, but an evaluation mismatch prevents the metric tracking script from registering true positive outcomes.

---

#### Optimization Step Convergence
![Gradient Descending Loss Track](figures/loss_curve.png)

> **Project Analysis & Observation (Loss Curve):**
> This line chart maps the Binary Cross-Entropy loss reduction of our custom Logistic Regression script over a span of 1000 gradient descent training iterations.
> 
> **Key Observation:** The curve experiences an aggressive drop over the first 200 epochs before smoothly stabilizing into a long, flat baseline past iteration 600. This stable decay curve proves that the chosen optimization learning rate ($\alpha = 0.01$) is balanced perfectly—allowing rapid gradient updates early on without inducing volatile oscillations or bouncing away from the global cost minimum.

---

## 8. DISCUSSION & DECISION BOUNDARIES

#### Geometric Feature Space Visualizations
![Geometric Separation Visual](figures/3_decision_boundary.png)

> **Project Analysis & Observation (Decision Boundary Scatter):**
> This scatter visualization showcases how each model splits the geometric feature space into distinct classification zones for Malignant and Benign diagnoses.
> 
> **Key Observation:** The plotted data points present a clear diagonal territory where benign and malignant markers begin to blend and overlap. Because Logistic Regression is mathematically locked into drawing a straight line, it slices clean through the center, leaving a few minor errors along the dividing line. The Decision Tree instead creates a jagged, staircase boundary that winds around specific point clusters. While this strict partitioning yields superior scores on our training data, this rigid boundary structure acts as a visual warning that the tree is likely overfitting to localized noise.

---

## 9. LIMITATIONS

### Dataset Limitations
* **Imbalance Vulnerability:** While minor, the 357 to 212 distribution can make accuracy misleading. If applied to highly skewed datasets, models could optimize for the majority class while failing to flag rare malignant tumors.

### Model Limitations
* **Logistic Regression:** Relies on a strict assumption of linear separability. If the feature boundaries are complex or non-linear, its performance degrades quickly.
* **Decision Tree:** Highly prone to overfitting when unconstrained. A depth of 5 can build rules too specific to the training data, hurting its ability to generalize.

### Training and Pipeline Limitations (The KNN Bug)
* The pipeline revealed a major structural issue with the custom KNN implementation. While it achieved an accuracy of $98.07\%$, its Precision, Recall, and Confusion Matrix collapsed to $0$.
* **Root Cause:** This issue points to a shape or type mismatch between the arrays inside `src/metrics.py` (e.g., comparing a 1D vector `(569,)` with a 2D vector `(569,1)`). This causes the Boolean logic for True Positives and True Negatives to break, evaluating to zero items.

## 10. CONCLUSION

### What was Learned
Building machine learning pipelines from scratch teaches you that model success relies heavily on exact data alignment. Even small mismatches in vector dimensions can break down evaluation metrics, even if the underlying model is mathematically sound.

### What was Proven Experimentally
The Decision Tree achieved the highest performance ($99.47\%$ accuracy, $1.0000$ precision), proving that a hierarchical approach works exceptionally well for this feature space. Logistic Regression followed closely at $98.24\%$, confirming that standard linear boundaries can effectively separate standardized cell data.

### What Remains Unclear
Without validation on a separate test set, it remains unclear how much of the Decision Tree's performance is due to overfitting its splits to the training data. Additionally, resolving the array shapes in the KNN evaluation is required to confirm its true performance on this dataset.